In [1]:
import pandas as pd
import datasets
from datasets import Image as Image_ds # change name because of similar PIL module
from datasets import Dataset
import os
import requests 
from PIL import Image
from tqdm import tqdm
import torch
from datasets import load_dataset
import numpy as np
import mteb

/Users/au672746/Library/CloudStorage/OneDrive-Aarhusuniversitet/CHC/art-multimodal-benchmark/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load dataset from HuggingFace
ds = load_dataset("louisebrix/smk_canon_paintings", split="train")
subset = ds.select(range(30))

In [3]:
leaderboard = pd.read_csv(os.path.join('data', 'mieb_leaderboard.csv'))

In [4]:
# get list of all model metas
metas = mteb.get_model_metas()

# filter out image models, save their names
vision_names = [meta.name for meta in metas if "image" in meta.modalities]

To match the model metas with leaderboard positions, we need to find the full model paths of the leaderboard models:

In [5]:
# get model names from column of names + HF links
leaderboard_model_names = []

for model_link in leaderboard['Model'].iloc[0:20]:
    model_name = model_link.split(']')[0][1:]
    leaderboard_model_names.append(model_name)

# match model name to full HF path
models_full_paths = []

for model in leaderboard_model_names:
    for path in vision_names:
        if path.endswith(model):
            models_full_paths.append(path)

model_metas = []
for model in models_full_paths:
    model_meta = mteb.get_model_meta(model)
    model_metas.append(dict(model_meta))

model_metadata = pd.DataFrame(model_metas)

In [6]:
model_metadata.head()

,name,revision,release_date,languages,loader,n_parameters,memory_usage_mb,max_tokens,embed_dim,license,...,public_training_data,framework,reference,similarity_fn_name,use_instructions,training_datasets,adapted_from,superseded_by,is_cross_encoder,modalities
0,QuanSun/EVA02-CLIP-bigE-14-plus,11afd202f2ae80869d6cef18b1ec775e79bd8d12,2023-04-26,[eng-Latn],functools.partial(<function evaclip_loader at ...,5000000000,19073.0,77.0,1024,mit,...,https://laion.ai/blog/laion-5b/,[PyTorch],https://huggingface.co/QuanSun/EVA-CLIP,None,False,{},None,None,None,"[image, text]"
1,laion/CLIP-ViT-bigG-14-laion2B-39B-b160k,bc7788f151930d91b58474715fdce5524ad9a189,2023-01-23,[eng-Latn],functools.partial(<function openclip_loader at...,2540000000,9689.0,77.0,1280,mit,...,https://laion.ai/blog/laion-5b/,[PyTorch],https://huggingface.co/laion/CLIP-ViT-bigG-14-...,None,False,{},None,None,None,"[image, text]"
2,QuanSun/EVA02-CLIP-bigE-14,11afd202f2ae80869d6cef18b1ec775e79bd8d12,2023-04-26,[eng-Latn],functools.partial(<function evaclip_loader at ...,4700000000,17929.0,77.0,1024,mit,...,https://laion.ai/blog/laion-5b/,[PyTorch],https://huggingface.co/QuanSun/EVA-CLIP,None,False,{},None,None,None,"[image, text]"
3,google/siglip-so400m-patch14-384,9fdffc58afc957d1a03a25b10dba0329ab15c2a3,2024-01-08,[eng-Latn],functools.partial(<class 'mteb.models.siglip_m...,878000000,3349.0,64.0,1152,apache-2.0,...,None,[PyTorch],https://huggingface.co/google/siglip-so400m-pa...,None,False,{},None,None,None,"[image, text]"
4,google/siglip-large-patch16-384,ce005573a40965dfd21fd937fbdeeebf2439fc35,2024-01-08,[eng-Latn],functools.partial(<class 'mteb.models.siglip_m...,652000000,2489.0,64.0,1024,apache-2.0,...,None,[PyTorch],https://huggingface.co/google/siglip-large-pat...,None,False,{},None,None,None,"[image, text]"


In [7]:
small_subset = ds.select(range(6))

In [8]:
def extract_embeddings(image, model_path):
    
    meta = mteb.get_model_meta(model_path)
    
    model = meta.load_model()
    
    embeddings = model.get_image_embeddings([image])
    
    return embeddings

In [11]:
from open_clip import create_model_and_transforms

In [ ]:
# test every model on 1 image to get not-installed dependecies..
for model_path in model_metadata['name']:
    try:
        embedding = extract_embeddings(small_subset[0]['image'], model_path)
        print(f"{model_path}: {embedding.shape}")
    except Exception as e:
        print(f'could not load model {model_path}: {e}')

could not load model QuanSun/EVA02-CLIP-bigE-14-plus: Please run `git clone git@github.com:baaivision/EVA.git`,`pip install ninja timm``pip install -v -U git+https://github.com/facebookresearch/xformers.git@main#egg=xformers``git clone https://github.com/NVIDIA/apex && cd apex && pip install -v --disable-pip-version-check --no-build-isolation --no-cache-dir ./`


/Users/au672746/Library/CloudStorage/OneDrive-Aarhusuniversitet/CHC/art-multimodal-benchmark/env/lib/python3.11/site-packages/mteb/models/openclip_models.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
/Users/au672746/Library/CloudStorage/OneDrive-Aarhusuniversitet/CHC/art-multimodal-benchmark/env/lib/python3.11/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
100%|██████████| 1/1 [00:11<00:00, 11.15s/it]


laion/CLIP-ViT-bigG-14-laion2B-39B-b160k: torch.Size([1, 1280])
could not load model QuanSun/EVA02-CLIP-bigE-14: Please run `git clone git@github.com:baaivision/EVA.git`,`pip install ninja timm``pip install -v -U git+https://github.com/facebookresearch/xformers.git@main#egg=xformers``git clone https://github.com/NVIDIA/apex && cd apex && pip install -v --disable-pip-version-check --no-build-isolation --no-cache-dir ./`


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|██████████| 1/1 [00:04<00:00,  4.12s/it]


google/siglip-so400m-patch14-384: torch.Size([1, 1152])


...

google/siglip-base-patch16-256-multilingual: torch.Size([1, 768])


100%|██████████| 1/1 [00:02<00:00,  2.68s/it]


openai/clip-vit-large-patch14: torch.Size([1, 768])


100%|██████████| 1/1 [00:02<00:00,  2.69s/it]

facebook/dinov2-large: torch.Size([1, 1024])
could not load model QuanSun/EVA02-CLIP-L-14: Please run `git clone git@github.com:baaivision/EVA.git`,`pip install ninja timm``pip install -v -U git+https://github.com/facebookresearch/xformers.git@main#egg=xformers``git clone https://github.com/NVIDIA/apex && cd apex && pip install -v --disable-pip-version-check --no-build-isolation --no-cache-dir ./`


In [15]:
import mteb
print(mteb.__file__)

/Users/au672746/Library/CloudStorage/OneDrive-Aarhusuniversitet/CHC/art-multimodal-benchmark/env/lib/python3.11/site-packages/mteb/__init__.py


In [19]:
embeddings_test = extract_embeddings(small_subset[0]['image'], 'openai/clip-vit-base-patch32')

openai/clip-vit-base-patch32


100%|██████████| 1/1 [00:00<00:00, 12.46it/s]


In [20]:
embeddings_test.shape

torch.Size([1, 512])

In [88]:
model_meta = pd.DataFrame([dict(mteb.get_model_meta(models_full_paths[0]))])
#testy = pd.DataFrame([model_meta])

In [89]:
model_meta

,name,revision,release_date,languages,loader,n_parameters,memory_usage_mb,max_tokens,embed_dim,license,...,public_training_data,framework,reference,similarity_fn_name,use_instructions,training_datasets,adapted_from,superseded_by,is_cross_encoder,modalities
0,QuanSun/EVA02-CLIP-bigE-14-plus,11afd202f2ae80869d6cef18b1ec775e79bd8d12,2023-04-26,[eng-Latn],functools.partial(<function evaclip_loader at ...,5000000000,19073.0,77.0,1024,mit,...,https://laion.ai/blog/laion-5b/,[PyTorch],https://huggingface.co/QuanSun/EVA-CLIP,None,False,{},None,None,None,"[image, text]"


In [85]:
testy.columns

Index(['name', 'revision', 'release_date', 'languages', 'loader',
       'n_parameters', 'memory_usage_mb', 'max_tokens', 'embed_dim', 'license',
       'open_weights', 'public_training_code', 'public_training_data',
       'framework', 'reference', 'similarity_fn_name', 'use_instructions',
       'training_datasets', 'adapted_from', 'superseded_by',
       'is_cross_encoder', 'modalities'],
      dtype='object')

In [86]:
testy

,name,revision,release_date,languages,loader,n_parameters,memory_usage_mb,max_tokens,embed_dim,license,...,public_training_data,framework,reference,similarity_fn_name,use_instructions,training_datasets,adapted_from,superseded_by,is_cross_encoder,modalities
0,QuanSun/EVA02-CLIP-bigE-14-plus,11afd202f2ae80869d6cef18b1ec775e79bd8d12,2023-04-26,[eng-Latn],functools.partial(<function evaclip_loader at ...,5000000000,19073.0,77.0,1024,mit,...,https://laion.ai/blog/laion-5b/,[PyTorch],https://huggingface.co/QuanSun/EVA-CLIP,None,False,{},None,None,None,"[image, text]"


In [81]:
mteb.get_model_metas(models_full_paths)

[ModelMeta(name='openai/clip-vit-large-patch14', revision='32bd64288804d66eefd0ccbe215aa642df71cc41', release_date='2021-02-26', languages=['eng-Latn'], loader=functools.partial(<class 'mteb.models.clip_models.CLIPModelWrapper'>, model_name='openai/clip-vit-large-patch14'), n_parameters=428000000, memory_usage_mb=1631.0, max_tokens=77.0, embed_dim=768, license=None, open_weights=True, public_training_code=None, public_training_data=None, framework=['PyTorch'], reference='https://huggingface.co/openai/clip-vit-large-patch14', similarity_fn_name=None, use_instructions=False, training_datasets=None, adapted_from=None, superseded_by=None, is_cross_encoder=None, modalities=['image', 'text']),
 ModelMeta(name='facebook/dinov2-large', revision='47b73eefe95e8d44ec3623f8890bd894b6ea2d6c', release_date='2023-07-18', languages=['eng-Latn'], loader=functools.partial(<class 'mteb.models.dino_models.DINOModelWrapper'>, model_name='facebook/dinov2-large'), n_parameters=304000000, memory_usage_mb=1161

In [10]:
import timm

In [11]:
timm.list_models('*eva*')

['eva02_base_patch14_224',
 'eva02_base_patch14_448',
 'eva02_base_patch16_clip_224',
 'eva02_enormous_patch14_clip_224',
 'eva02_large_patch14_224',
 'eva02_large_patch14_448',
 'eva02_large_patch14_clip_224',
 'eva02_large_patch14_clip_336',
 'eva02_small_patch14_224',
 'eva02_small_patch14_336',
 'eva02_tiny_patch14_224',
 'eva02_tiny_patch14_336',
 'eva_giant_patch14_224',
 'eva_giant_patch14_336',
 'eva_giant_patch14_560',
 'eva_giant_patch14_clip_224',
 'eva_large_patch14_196',
 'eva_large_patch14_336']